In [ ]:
from agent import getAgent

agent = getAgent()

dataset = [
    {"id": 1, "category": "leave_balance",
     "question": "I've used 11 earned leaves this year. How many are left?",
     "expected_tools": ["leave_balance"],
     "expected_args": {"leave_balance": {"leave_type": "earned", "days_taken": 11}},
     "reference": "7 days remaining (18 - 11)"}
     ]

config = {"configurable":{"thread_id":"user123"}}

In [ ]:
from langchain_core.messages import AIMessage,ToolMessage
def extract(messages):
    calls , outputTool =[], {}
    for msg in messages:
        if isinstance(msg,AIMessage):
            calls.extend(msg.tool_calls)
        elif isinstance(msg,ToolMessage):
            outputTool[msg.tool_call_id] = str(msg.content).lower()
    return calls, outputTool


In [ ]:
data = dataset[0]
messages = agent.invoke(input = {"messages":[{"role":"user","content":data["question"]}]},config=config)["messages"]

In [ ]:
calls , output = extract(messages)
count = dict(n_calls=0, hit_tools = 0, hit_arg = 0, hit_answer = 0, hit_message = 0)

In [ ]:
exp = data["expected_tools"]
called_tools = [t["name"] for t in calls]
if set(exp) <= set(called_tools):
    print("ALl called")
    count["hit_tools"] += 1

In [ ]:
#check args
dataset1= [
    {"id": 1, "category": "leave_balance",
     "question": "I've used 11 earned leaves this year. How many are left?",
     "expected_tools": ["leave_balance"],
     "expected_args": {"leave_balance": {"leave_type": "earned", "days_taken": 11}},
     "reference": "7 days remaining (18 - 11)"}, 
     {"id": 18, "category": "multi_tool",
     "question": "Is there a limit on how many casual leaves I can take in a row? Also, I've used 6 casual leaves, how many are left?",
     "expected_tools": ["search_hr_policy", "leave_balance"],
     "expected_args": {"leave_balance": {"leave_type": "casual", "days_taken": 6}},
     "reference": "Maximum 3 consecutive days; 2 casual leaves remaining"}
     ]

In [ ]:
calls

In [ ]:
def normalize(v):
    return " ".join(str(v).lower().split())

In [ ]:
def arg_match(actual_args, expected_args):
    for exp_key, exp_val in expected_args.items():
        print(exp_key,exp_val,actual_args)
        if exp_key not in actual_args:
            return False
        act_val , exp = normalize(actual_args[exp_key]),normalize(exp_val)
        if isinstance(exp_val,str):
            if not all(w in act_val for w in exp.split()):
                return False
        elif act_val != exp:
            # numbers, booleans, enums: exact match
            return False
    return True

In [ ]:
def arg_check(calls,exp_name, exp_args):
    return any(tc["name"]==exp_name and arg_match(tc["args"],exp_args) for tc in calls)
    # return any(tc["name"] == exp_name and all(t in (tc["args"].keys()) for t in exp_args) for tc in calls)
    



In [ ]:
if all(arg_check(calls, exp_name, exp_args) for exp_name, exp_args in dataset[0]["expected_args"].items()):
    hit_args = True